# Deep dive 10/13: `satisfaction_surveys`

Dictionary declares 250,000 rows, Source: Contact Center, Partition: daily. Every column, Bronze and Silver.

`table_specs.py`'s comment makes a specific claim worth verifying directly rather than trusting: **`main_score` is described as "a 1-7 integer scale spanning CSAT/NPS/CES"**, but the dictionary itself describes it differently per survey type -- 1-5 for CSAT, 0-10 for NPS (CES isn't given an explicit range in the dictionary text). Section 7 checks the real observed range of `main_score`, broken out by `survey_type`, to see which description -- if either -- actually matches the data.

Dictionary NOT NULL: `survey_id`, `survey_date`, `process_date`, `customer_id`, `survey_type`, `send_channel`, `main_score` -- **7 columns**. `contracts.py`'s current `required` tuple has 6 -- **`send_channel` is missing**, checked in section 3 as a fix candidate. Nullable: `interaction_id`, `agent_id` (both nullable FKs -- not every survey ties back to a specific interaction or agent), `nps_category`, `question_1_text`/`question_1_response`, `question_2_text`/`question_2_response`, `question_3_text`/`question_3_response`, `open_comments`, `comment_sentiment`, `response_time_hours`, `campaign_response_rate`.

Since `call_center_interactions` is already checked, section 11 reuses the cross-table consistency pattern from `call_transcripts`: when `interaction_id` is populated, does the survey's `customer_id`/`agent_id` actually match the parent interaction's own values?

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 0. Sample rows -- Bronze and Silver side by side (excluding long text columns)

In [2]:
sample_ids = q("SELECT survey_id FROM bronze.satisfaction_surveys ORDER BY survey_id LIMIT 20")["survey_id"].tolist()
ids_sql = ", ".join(f"'{i}'" for i in sample_ids)

cols = ("survey_id, survey_date, process_date, interaction_id, customer_id, agent_id, survey_type, "
        "send_channel, main_score, nps_category, question_1_response, question_2_response, "
        "question_3_response, comment_sentiment, response_time_hours, campaign_response_rate")
print("Bronze:")
display(q(f"SELECT {cols} FROM bronze.satisfaction_surveys WHERE survey_id IN ({ids_sql}) ORDER BY survey_id"))
print("Silver:")
display(q(f"SELECT {cols} FROM silver.fact_satisfaction_surveys WHERE survey_id IN ({ids_sql}) ORDER BY survey_id"))

Bronze:


,survey_id,survey_date,process_date,interaction_id,customer_id,agent_id,survey_type,send_channel,main_score,nps_category,question_1_response,question_2_response,question_3_response,comment_sentiment,response_time_hours,campaign_response_rate
0,SRV-000056NJ558MNADQ0EZK,2023-12-11 21:01:08,2023-12-11,INT-MGRAY892ZHPNRWUQ,CLI-1SY88987YM9K,AGT-ZJEV0XGD00,CSAT,App,3,NaN,5.0,NaN,5.0,NaN,21.06,37.24
1,SRV-0002NSR3NFAZFIIXIKSS,2023-11-04 06:49:14,2023-11-03,INT-XQN7UD3J5I6B0Y6X,CLI-UPSWLBV0SJSQ,AGT-0SFUB4MUFS,CSAT,SMS,3,NaN,3.0,2.0,3.0,NaN,21.69,32.25
2,SRV-00039DRVU0F5DN0TNO57,2025-04-15 09:27:14,2025-04-14,INT-O74LTAYVZPV6RMHK,CLI-8SGS4HALBZ76,AGT-W4AYW4BGI9,NPS,IVR,7,Passive,NaN,NaN,4.0,Neutral,25.47,22.59
3,SRV-000FT7X8R1ZAC158FBDO,2026-05-06 13:03:33,2026-05-05,INT-J2PWGP7G52K0B4HA,CLI-SXY90QATQK3E,AGT-EKGNQCKBSX,NPS,SMS,5,Detractor,5.0,2.0,NaN,NaN,18.62,33.49
4,SRV-000HOYC3SRMW0F33VWSS,2024-09-16 03:08:29,2024-09-15,INT-89GJ0KGZRLL834GE,CLI-HBCIMP8BPCNJ,AGT-KU6FVZPQSV,CES,App,3,NaN,NaN,NaN,NaN,NaN,12.97,43.07
5,SRV-000I3AGOE9KO2N2Q2APU,2025-01-15 11:00:57,2025-01-14,INT-JXG3L1AL3Y5OCPYD,CLI-W6YK1RM8TSMW,AGT-ROT1G7EATP,NPS,Email,5,Detractor,2.0,NaN,NaN,NaN,29.03,29.8
6,SRV-000MNEGD4LW8YGX367VJ,2026-03-16 23:46:27,2026-03-16,INT-C5B7S538Y0F0QJUT,CLI-SNRYOEDF4L6J,AGT-0SK7J9FAUM,CSAT,Email,2,NaN,NaN,NaN,5.0,NaN,16.86,19.42
7,SRV-000N643DJZXY9CP8OWHH,2025-09-06 11:47:42,2025-09-06,INT-JP1T20ODPH4NKPC8,CLI-LSJ5YCTWL58M,AGT-H600EXWVP2,CSAT,Email,3,NaN,1.0,4.0,NaN,Negative,9.47,44.43
8,SRV-000OYZA3XQLU48GJKPUK,2024-01-05 00:45:32,2024-01-04,INT-NBYJB6NOODDZ2HVJ,CLI-CGB0GWWPPVI8,AGT-56XIRH3EBF,CSAT,App,2,NaN,NaN,3.0,NaN,Negative,17.36,23.14
9,SRV-000TF1KFO283MBBSILW3,2024-08-08 15:17:43,2024-08-07,INT-CHEK907OAQ2TB5W2,CLI-5R500MDZD4R5,AGT-HEY1DFYU3O,CSAT,Web,3,NaN,NaN,NaN,NaN,NaN,15.99,25.23


Silver:


,survey_id,survey_date,process_date,interaction_id,customer_id,agent_id,survey_type,send_channel,main_score,nps_category,question_1_response,question_2_response,question_3_response,comment_sentiment,response_time_hours,campaign_response_rate
0,SRV-000056NJ558MNADQ0EZK,2023-12-11 21:01:08,2023-12-11,INT-MGRAY892ZHPNRWUQ,CLI-1SY88987YM9K,AGT-ZJEV0XGD00,CSAT,App,3,NaN,5.0,NaN,5.0,NaN,21.06,37.24
1,SRV-0002NSR3NFAZFIIXIKSS,2023-11-04 06:49:14,2023-11-03,INT-XQN7UD3J5I6B0Y6X,CLI-UPSWLBV0SJSQ,AGT-0SFUB4MUFS,CSAT,SMS,3,NaN,3.0,2.0,3.0,NaN,21.69,32.25
2,SRV-00039DRVU0F5DN0TNO57,2025-04-15 09:27:14,2025-04-14,INT-O74LTAYVZPV6RMHK,CLI-8SGS4HALBZ76,AGT-W4AYW4BGI9,NPS,IVR,7,Passive,NaN,NaN,4.0,Neutral,25.47,22.59
3,SRV-000FT7X8R1ZAC158FBDO,2026-05-06 13:03:33,2026-05-05,INT-J2PWGP7G52K0B4HA,CLI-SXY90QATQK3E,AGT-EKGNQCKBSX,NPS,SMS,5,Detractor,5.0,2.0,NaN,NaN,18.62,33.49
4,SRV-000HOYC3SRMW0F33VWSS,2024-09-16 03:08:29,2024-09-15,INT-89GJ0KGZRLL834GE,CLI-HBCIMP8BPCNJ,AGT-KU6FVZPQSV,CES,App,3,NaN,NaN,NaN,NaN,NaN,12.97,43.07
5,SRV-000I3AGOE9KO2N2Q2APU,2025-01-15 11:00:57,2025-01-14,INT-JXG3L1AL3Y5OCPYD,CLI-W6YK1RM8TSMW,AGT-ROT1G7EATP,NPS,Email,5,Detractor,2.0,NaN,NaN,NaN,29.03,29.80
6,SRV-000MNEGD4LW8YGX367VJ,2026-03-16 23:46:27,2026-03-16,INT-C5B7S538Y0F0QJUT,CLI-SNRYOEDF4L6J,AGT-0SK7J9FAUM,CSAT,Email,2,NaN,NaN,NaN,5.0,NaN,16.86,19.42
7,SRV-000N643DJZXY9CP8OWHH,2025-09-06 11:47:42,2025-09-06,INT-JP1T20ODPH4NKPC8,CLI-LSJ5YCTWL58M,AGT-H600EXWVP2,CSAT,Email,3,NaN,1.0,4.0,NaN,Negative,9.47,44.43
8,SRV-000OYZA3XQLU48GJKPUK,2024-01-05 00:45:32,2024-01-04,INT-NBYJB6NOODDZ2HVJ,CLI-CGB0GWWPPVI8,AGT-56XIRH3EBF,CSAT,App,2,NaN,NaN,3.0,NaN,Negative,17.36,23.14
9,SRV-000TF1KFO283MBBSILW3,2024-08-08 15:17:43,2024-08-07,INT-CHEK907OAQ2TB5W2,CLI-5R500MDZD4R5,AGT-HEY1DFYU3O,CSAT,Web,3,NaN,NaN,NaN,NaN,NaN,15.99,25.23


## 1. Column inventory -- names AND types, Bronze vs. Silver

In [3]:
expected_silver_columns = [
    "survey_id", "survey_date", "process_date", "interaction_id", "customer_id", "agent_id",
    "survey_type", "send_channel", "main_score", "nps_category", "question_1_text",
    "question_1_response", "question_2_text", "question_2_response", "question_3_text",
    "question_3_response", "open_comments", "comment_sentiment", "response_time_hours",
    "campaign_response_rate",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_satisfaction_surveys'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.fact_satisfaction_surveys:", missing or "none")
print("Extra in silver.fact_satisfaction_surveys:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.fact_satisfaction_surveys: none
Extra in silver.fact_satisfaction_surveys: none

Column order as built: ['survey_id', 'survey_date', 'process_date', 'interaction_id', 'customer_id', 'agent_id', 'survey_type', 'send_channel', 'main_score', 'nps_category', 'question_1_text', 'question_1_response', 'question_2_text', 'question_2_response', 'question_3_text', 'question_3_response', 'open_comments', 'comment_sentiment', 'response_time_hours', 'campaign_response_rate']


In [4]:
print("Bronze columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='satisfaction_surveys' ORDER BY ordinal_position
"""))

print("Silver columns + types:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='fact_satisfaction_surveys' ORDER BY ordinal_position
"""))

Bronze columns + types:


,column_name,data_type
0,survey_id,VARCHAR
1,survey_date,VARCHAR
2,process_date,VARCHAR
3,interaction_id,VARCHAR
4,customer_id,VARCHAR
5,agent_id,VARCHAR
6,survey_type,VARCHAR
7,send_channel,VARCHAR
8,main_score,VARCHAR
9,nps_category,VARCHAR


Silver columns + types:


,column_name,data_type
0,survey_id,VARCHAR
1,survey_date,TIMESTAMP
2,process_date,DATE
3,interaction_id,VARCHAR
4,customer_id,VARCHAR
5,agent_id,VARCHAR
6,survey_type,VARCHAR
7,send_channel,VARCHAR
8,main_score,BIGINT
9,nps_category,VARCHAR


## 2. Row-count reconciliation + dedup

In [5]:
bronze_total = q("SELECT count(*) AS n FROM bronze.satisfaction_surveys")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT survey_id) AS n FROM bronze.satisfaction_surveys")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.fact_satisfaction_surveys")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT survey_id) AS n FROM silver.fact_satisfaction_surveys")["n"][0]

print(f"bronze.satisfaction_surveys:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct survey_id")
print(f"silver.fact_satisfaction_surveys: {silver_total:,} rows, {silver_distinct_pk:,} distinct survey_id")
print()
print("Dictionary declares 250,000 rows -- matches" if bronze_total == 250_000 else f"Dictionary declares 250,000 rows -- MISMATCH, Bronze has {bronze_total:,}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.satisfaction_surveys:      212,759 rows, 212,759 distinct survey_id
silver.fact_satisfaction_surveys: 212,759 rows, 212,759 distinct survey_id

Dictionary declares 250,000 rows -- MISMATCH, Bronze has 212,759
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit -- Bronze raw vs. Silver typed, all 7 dictionary-declared columns

Including `send_channel`, missing from `contracts.py`'s `required` tuple -- checked here as a fix candidate.

In [6]:
declared_not_null = ["survey_id", "survey_date", "process_date", "customer_id", "survey_type", "send_channel", "main_score"]
currently_required = {"survey_id","survey_date","process_date","customer_id","survey_type","main_score"}

rows = []
for col in declared_not_null:
    bronze_nulls = q(f"""
        SELECT count(*) AS n FROM bronze.satisfaction_surveys
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    silver_nulls = q(f"""
        SELECT count(*) AS n FROM silver.fact_satisfaction_surveys WHERE {col} IS NULL
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "in_contracts_required": col in currently_required,
                 "bronze_nulls": bronze_nulls, "silver_nulls": silver_nulls,
                 "cast_introduced_nulls": silver_nulls - bronze_nulls})

pd.DataFrame(rows)

,column,declared,in_contracts_required,bronze_nulls,silver_nulls,cast_introduced_nulls
0,survey_id,NOT NULL,True,0,0,0
1,survey_date,NOT NULL,True,0,0,0
2,process_date,NOT NULL,True,0,0,0
3,customer_id,NOT NULL,True,0,0,0
4,survey_type,NOT NULL,True,0,0,0
5,send_channel,NOT NULL,False,0,0,0
6,main_score,NOT NULL,True,0,0,0


## 4. Nullable columns -- population rate, Bronze vs. Silver

In [7]:
nullable_cols = ["interaction_id", "agent_id", "nps_category", "question_1_text", "question_1_response",
                  "question_2_text", "question_2_response", "question_3_text", "question_3_response",
                  "open_comments", "comment_sentiment", "response_time_hours", "campaign_response_rate"]
rows = []
total = q("SELECT count(*) AS n FROM bronze.satisfaction_surveys")["n"][0]
for col in nullable_cols:
    bronze_populated = q(f"""
        SELECT count(*) AS n FROM bronze.satisfaction_surveys
        WHERE {col} IS NOT NULL AND trim(CAST({col} AS VARCHAR)) <> ''
    """)["n"][0]
    silver_populated = q(f"""
        SELECT count(*) AS n FROM silver.fact_satisfaction_surveys WHERE {col} IS NOT NULL
    """)["n"][0]
    rows.append({"column": col, "total": total,
                 "bronze_populated": bronze_populated, "bronze_pct": round(100*bronze_populated/total, 2),
                 "silver_populated": silver_populated, "silver_pct": round(100*silver_populated/total, 2)})

pd.DataFrame(rows)

,column,total,bronze_populated,bronze_pct,silver_populated,silver_pct
0,interaction_id,212759,212759,100.00,212759,100.00
1,agent_id,212759,212759,100.00,212759,100.00
2,nps_category,212759,60394,28.39,60394,28.39
3,question_1_text,212759,121303,57.01,121303,57.01
4,question_1_response,212759,121370,57.05,121370,57.05
5,question_2_text,212759,81371,38.25,81371,38.25
6,question_2_response,212759,81496,38.30,81496,38.30
7,question_3_text,212759,40144,18.87,40144,18.87
8,question_3_response,212759,40103,18.85,40103,18.85
9,open_comments,212759,101196,47.56,101196,47.56


## 5. Domain checks -- `survey_type`, `send_channel`, `nps_category`

In [8]:
# survey_type -- dictionary: CSAT, NPS, CES
print("Bronze:")
display(q("SELECT survey_type, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC"))
# send_channel -- dictionary: Email, SMS, IVR, App, Web
print("send_channel:")
display(q("SELECT send_channel, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC"))
# nps_category -- dictionary: Promoter, Passive, Detractor
print("nps_category:")
display(q("SELECT nps_category, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC"))

Bronze:


,survey_type,n
0,CSAT,127856
1,NPS,63668
2,CES,21235


send_channel:


,send_channel,n
0,Email,84880
1,SMS,63798
2,App,42595
3,IVR,10836
4,Web,10650


nps_category:


,nps_category,n
0,NaN,152365
1,Detractor,45007
2,Passive,15387


## 6. FK integrity -- `interaction_id`/`agent_id` (nullable), `customer_id` (required), reproduced directly

In [9]:
fk_checks = [
    ("interaction_id", "silver.fact_call_center_interactions", "interaction_id"),
    ("customer_id", "silver.dim_customers", "customer_id"),
    ("agent_id", "silver.dim_service_agents", "agent_id"),
]
rows = []
for child_col, parent_table, parent_col in fk_checks:
    total = q(f"SELECT count(*) AS n FROM silver.fact_satisfaction_surveys WHERE {child_col} IS NOT NULL")["n"][0]
    orphans = q(f"""
        SELECT count(*) AS n FROM silver.fact_satisfaction_surveys t
        WHERE t.{child_col} IS NOT NULL
          AND NOT EXISTS (SELECT 1 FROM {parent_table} p WHERE p.{parent_col} = t.{child_col})
    """)["n"][0]
    rows.append({"fk": f"{child_col} -> {parent_table}.{parent_col}", "populated": total,
                 "orphans": orphans, "orphan_pct": round(100*orphans/total, 4) if total else None})

pd.DataFrame(rows)

,fk,populated,orphans,orphan_pct
0,interaction_id -> silver.fact_call_center_interactions.interaction_id,212759,0,0.0
1,customer_id -> silver.dim_customers.customer_id,212759,0,0.0
2,agent_id -> silver.dim_service_agents.agent_id,212759,0,0.0


## 7. `main_score` -- real observed range BY `survey_type`

`table_specs.py`'s comment claims "a 1-7 integer scale spanning CSAT/NPS/CES." The dictionary instead says 1-5 for CSAT and 0-10 for NPS (no explicit range given for CES). Checking which description -- if either -- the real data actually matches.

In [10]:
q("""
    SELECT survey_type, count(*) AS n, min(main_score) AS min_score, max(main_score) AS max_score,
           count(DISTINCT main_score) AS distinct_values
    FROM silver.fact_satisfaction_surveys
    GROUP BY 1
    ORDER BY 1
""")

,survey_type,n,min_score,max_score,distinct_values
0,CES,21235,1,4,4
1,CSAT,127856,1,4,4
2,NPS,63668,2,7,6


In [11]:
# Full distribution per survey_type, to see the actual shape rather than just min/max.
for t in q("SELECT DISTINCT survey_type AS v FROM silver.fact_satisfaction_surveys")["v"]:
    print(f"{t}:")
    display(q(f"SELECT main_score, count(*) AS n FROM silver.fact_satisfaction_surveys WHERE survey_type = '{t}' GROUP BY 1 ORDER BY 1"))

NPS:


,main_score,n
0,2,4858
1,3,4900
2,4,4876
3,5,16475
4,6,16329
5,7,16230


CSAT:


,main_score,n
0,1,4422
1,2,35646
2,3,73313
3,4,14475


CES:


,main_score,n
0,1,737
1,2,5894
2,3,12150
3,4,2454


## 8. `nps_category` -- does it line up with `main_score` the standard NPS way (Promoter/Passive/Detractor)?

In [12]:
q("""
    SELECT nps_category, min(main_score) AS min_score, max(main_score) AS max_score, count(*) AS n
    FROM silver.fact_satisfaction_surveys
    WHERE survey_type = 'NPS' AND nps_category IS NOT NULL
    GROUP BY 1
    ORDER BY min_score
""")

,nps_category,min_score,max_score,n
0,Detractor,2,6,45007
1,Passive,7,7,15387


## 9. `question_1_response`/`question_2_response`/`question_3_response` -- range sanity (dictionary: 1-5 each)

In [13]:
q("""
    SELECT
        min(question_1_response) AS q1_min, max(question_1_response) AS q1_max,
        min(question_2_response) AS q2_min, max(question_2_response) AS q2_max,
        min(question_3_response) AS q3_min, max(question_3_response) AS q3_max,
        count(*) FILTER (WHERE question_1_response NOT BETWEEN 1 AND 5) AS q1_out_of_range,
        count(*) FILTER (WHERE question_2_response NOT BETWEEN 1 AND 5) AS q2_out_of_range,
        count(*) FILTER (WHERE question_3_response NOT BETWEEN 1 AND 5) AS q3_out_of_range
    FROM silver.fact_satisfaction_surveys
""")

,q1_min,q1_max,q2_min,q2_max,q3_min,q3_max,q1_out_of_range,q2_out_of_range,q3_out_of_range
0,1.0,5.0,1.0,5.0,1.0,5.0,0,0,0


## 10. `response_time_hours`/`campaign_response_rate` -- range sanity

In [14]:
q("""
    SELECT
        typeof(response_time_hours) AS rt_type, typeof(campaign_response_rate) AS crr_type,
        min(response_time_hours) AS min_rt, max(response_time_hours) AS max_rt,
        min(campaign_response_rate) AS min_crr, max(campaign_response_rate) AS max_crr,
        count(*) FILTER (WHERE response_time_hours < 0) AS negative_rt,
        count(*) FILTER (WHERE campaign_response_rate < 0 OR campaign_response_rate > 100) AS crr_out_of_range
    FROM silver.fact_satisfaction_surveys
    GROUP BY ALL
""")

,rt_type,crr_type,min_rt,max_rt,min_crr,max_crr,negative_rt,crr_out_of_range
0,DOUBLE,DOUBLE,1.01,35.97,15.0,45.0,0,0


## 11. Cross-table consistency -- when `interaction_id` is populated, does `customer_id`/`agent_id` match the parent interaction's own?

In [15]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE s.customer_id = i.customer_id) AS customer_id_matches,
        count(*) FILTER (WHERE s.customer_id <> i.customer_id) AS customer_id_mismatches,
        count(*) FILTER (WHERE s.agent_id IS NOT NULL AND i.agent_id IS NOT NULL AND s.agent_id = i.agent_id) AS agent_id_matches,
        count(*) FILTER (WHERE s.agent_id IS NOT NULL AND i.agent_id IS NOT NULL AND s.agent_id <> i.agent_id) AS agent_id_mismatches
    FROM silver.fact_satisfaction_surveys s
    JOIN silver.fact_call_center_interactions i ON i.interaction_id = s.interaction_id
    WHERE s.interaction_id IS NOT NULL
""")

,total,customer_id_matches,customer_id_mismatches,agent_id_matches,agent_id_mismatches
0,212759,212759,0,212759,0


## 12. `comment_sentiment` -- domain check

In [16]:
q("SELECT comment_sentiment, count(*) AS n FROM bronze.satisfaction_surveys GROUP BY 1 ORDER BY n DESC")

,comment_sentiment,n
0,NaN,111502
1,Negative,67529
2,Neutral,26774
3,Positive,6954


## 13. `survey_date`/`process_date` -- cast correctness, window check

In [17]:
q("""
    SELECT
        typeof(survey_date) AS survey_date_type, typeof(process_date) AS process_date_type,
        min(survey_date) AS earliest_survey, max(survey_date) AS latest_survey,
        min(process_date) AS earliest_process, max(process_date) AS latest_process,
        count(*) FILTER (WHERE process_date <> CAST(survey_date AS DATE)) AS process_date_differs
    FROM silver.fact_satisfaction_surveys
    GROUP BY ALL
""")

,survey_date_type,process_date_type,earliest_survey,latest_survey,earliest_process,latest_process,process_date_differs
0,TIMESTAMP,DATE,2023-06-17 09:29:20,2026-06-19 06:54:58,2023-06-17,2026-06-17,169092


In [18]:
q("""
    SELECT
        count(*) FILTER (WHERE survey_date < TIMESTAMP '2023-06-17') AS before_window,
        count(*) FILTER (WHERE survey_date > TIMESTAMP '2026-06-17 23:59:59') AS after_window
    FROM silver.fact_satisfaction_surveys
""")

,before_window,after_window
0,0,153


## Checklist

- [ ] Column inventory (1): all 19 Silver columns present, types correct
- [ ] Row reconciliation (2): 250,000 rows as the dictionary declares, dedup clean
- [ ] NOT NULL audit (3): all 7 declared columns populated -- note `send_channel` fix candidate
- [ ] Nullable columns (4): population rates make sense
- [ ] Domain checks (5): `survey_type`/`send_channel`/`nps_category` real values
- [ ] FK integrity (6): all three FKs reproduced directly
- [ ] `main_score` (7): resolves whether the real per-survey_type range matches the dictionary, the code comment, both, or neither
- [ ] `nps_category` (8): lines up with `main_score` the standard NPS way
- [ ] Question responses (9): within 1-5
- [ ] `response_time_hours`/`campaign_response_rate` (10): no negatives, rate within 0-100
- [ ] Cross-table consistency (11): survey `customer_id`/`agent_id` matches the parent interaction's, when populated
- [ ] `comment_sentiment` (12): domain sanity
- [ ] Dates (13): window check

In [20]:
con.close()
print("closed")

closed
